In [3]:
# I didn't measure launch success so I will use (as close as I can)
# to the industry standard of 50% visibility score, 40% race performance,
# and 10% for special events (playoffs etc)

# since colinearity for finish position and laps led is incredibly high 
# I will just be using laps led, because it had stronger correlation with visibility 

# Define weight configuration
WEIGHT_CONFIG = {
    'category_weights': {
        'laps_led': 0.40,    # 40% of total score
        'visibility_score': 0.50,      # 50% of total score
        'special_events': 0.10       # 10% of total score
    },
    'variable_weights_within_category': {
        'laps_led': {
            'laps_led': 1.00
        },
        'special_events': {
            'is_win': 0.60,           # 60% of special events weight
            'is_playoff': 0.40        # 40% of special events weight
        },
        'visibility_score': {
            'visibility_score': 1.00
    }
}
}
# Calculate effective weights for each variable
def calculate_effective_weights(config):
    """
    Calculate the final weight for each variable.

    Parameters:
    -----------
    config : dict
        Weight configuration with category and variable weights

    Returns:
    --------
    dict : Effective weight for each variable
    """
    effective = {}
    for category, cat_weight in config['category_weights'].items():
        var_weights = config['variable_weights_within_category'].get(category, {})
        for var, var_weight in var_weights.items():
            effective[var] = cat_weight * var_weight
    return effective

effective_weights = calculate_effective_weights(WEIGHT_CONFIG)

print("Effective Variable Weights:")
print("="*50)
for var, weight in sorted(effective_weights.items(), key=lambda x: -x[1]):
    print(f"{var:<30}: {weight*100:>5.1f}%")
print(f"{'TOTAL':<30}: {sum(effective_weights.values())*100:>5.1f}%")

Effective Variable Weights:
visibility_score              :  50.0%
laps_led                      :  40.0%
is_win                        :   6.0%
is_playoff                    :   4.0%
TOTAL                         : 100.0%


In [5]:
weight_rationale = """
# Visibility Score: Weighting Rationale

## Overall Philosophy

The weighting scheme prioritizes **on-track performance and media exposure** because:

1. On-track results determine visibility opportunity (better finishes = more camera time)
2. Traditional media still reaches the largest audience for NASCAR content
3. Social engagement, while valuable, represents a subset of total audience

## Category Rationale

### Race Performance (40%)

*Why 40%?* Race performance is the foundation of sponsorship value. A sponsor on a winning car gets exponentially more exposure than one on a mid-pack car. Our EDA showed that finish position correlates strongly with all visibility metrics.

**Variable breakdown:**
- **Laps Led (100%)**: Laps led is a strong proxy for race dominance
### Visibility Score (50%)
*Why 50%?* Visibility score is a composite metric that captures the total exposure across all media channels. It is the most direct measure of how much attention a driver and their sponsors receive.
**Variable breakdown:**
- **Visibility Score (100%)**: This is the aggregated score from all visibility sources,
### Special Events (10%)

*Why 10%?* Wins and playoff races generate disproportionate visibility spikes that should be captured.

**Variable breakdown:**
- **Wins (60%)**: A win generates 3-5x normal visibility. Even a small bonus weight has large impact when activated.
- **Playoffs (40%)**: Playoff races have higher stakes and more viewers. Built-in visibility multiplier.

## Design Decisions

### Why not equal weights?

Equal weights (14.3% each for 7 variables) would:
- Treat Reddit mentions as equally important as race wins
- Ignore the hierarchical importance of performance vs. social

### Why not data-driven weights from regression?

Regression-based weights would:
- Require a labeled "visibility outcome" variable (which doesn't exist)
- Be sensitive to sample size and outliers
- Be harder to explain to stakeholders

The hybrid approach provides transparency and defensibility.

## Sensitivity Testing

We will test alternative weight configurations in Substep 4.3.3:
- Higher performance weight (50/25/15/10)
- Higher media weight (30/40/20/10)
- Equal weights (25/25/25/25)

If rankings change significantly under alternative weights, we'll note the sensitivity in our methodology.
"""

# Save rationale document
with open('weighting_rationale.md', 'w') as f:
    f.write(weight_rationale)

print("Weighting rationale saved to: weighting_rationale.md")

Weighting rationale saved to: weighting_rationale.md


In [8]:
import json

# Full configuration for scoring model
SCORING_CONFIG = {
    'version': '1.0',
    'created_date': '2026-07-10',  # Update with actual date
    'category_weights': {
        'race_performance': 0.40,
        'visibility_score': 0.50,
        'special_events': 0.10
    },
    'variables': {
        'finish_position': {
            'category': 'race_performance',
            'weight_within_category': 0,
            'effective_weight': 0,
            'direction': 'inverse',
            'transform': 'invert_position'
        },
        'laps_led': {
            'category': 'race_performance',
            'weight_within_category': 1,
            'effective_weight': 0.4,
            'direction': 'direct',
            'transform': 'normalize'
        },
        'news_weighted_mentions': {
            'category': 'visibility_score',
            'weight_within_category': 'N/A',  # Placeholder for actual weight
            'effective_weight': 'N/A',  # Placeholder for actual effective weight
            'direction': 'direct',
            'transform': 'normalize'
        },
        'reddit_mentions': {
            'category': 'visibility_score',
            'weight_within_category': 'N/A',  # Placeholder for actual weight
            'effective_weight': 'N/A',  # Placeholder for actual effective weight
            'direction': 'direct',
            'transform': 'normalize'
        },
        'youtube_sponsor_views': {
            'category': 'visibility_score',
            'weight_within_category': 'N/A',  # Placeholder for actual weight
            'effective_weight': 'N/A',  # Placeholder for actual effective weight
            'direction': 'direct',
            'transform': 'normalize'
        },
        'is_win': {
            'category': 'special_events',
            'weight_within_category': 0.60,
            'effective_weight': 0.06,
            'direction': 'direct',
            'transform': 'binary'
        },
        'is_playoff': {
            'category': 'special_events',
            'weight_within_category': 0.40,
            'effective_weight': 0.04,
            'direction': 'direct',
            'transform': 'binary'
        }
    }
}

# Save configuration
with open('scoring_config.json', 'w') as f:
    json.dump(SCORING_CONFIG, f, indent=2)

print("Scoring configuration saved to: scoring_config.json")

Scoring configuration saved to: scoring_config.json
